In [1]:
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from ultralytics import YOLO, SAM 

In [2]:
from ultralytics import YOLO

if __name__ == '__main__':
    model = YOLO(r'E:\mastercode\6.yolo\ultralytics-main\ultralytics\cfg\models\11\yolo11-seg_DW.yaml')  # 可换 s/m/l/x
    model.train(
        data=r'E:\\mastercode\\6.yolo\\ultralytics-main\\voc20007_seg_yolov8.yaml',
        project=r'E:/mastercode/6.yolo/runs/segment',
        epochs=300,
        imgsz=640,
        batch=8,
        # lr0 = 0.00001,
        # lrf = 0.01,
        # momentum = 0.73,
        # weight_decay = 0.0005,
        # warmup_epochs = 3,
        # optimizer='AdamW',
        device=0  # CPU 则改为 'cpu'
    )

WARNING no model scale passed. Assuming scale='n'.


New https://pypi.org/project/ultralytics/8.4.21 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.19  Python-3.11.14 torch-2.5.0+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=E:\\mastercode\\6.yolo\\ultralytics-main\\voc20007_seg_yolov8.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=E:\mastercode\6.yol

In [2]:
from ultralytics import YOLO
from ultralytics.models.yolo.segment import SegmentationTrainer
from ultralytics.cfg import get_cfg, DEFAULT_CFG
from FEM_DCN import replace_c2f_with_fem, collect_fem_loss


class FEMSegmentationTrainer(SegmentationTrainer):

    def __init__(self, cfg=DEFAULT_CFG, overrides=None, _callbacks=None):
        if overrides is None:
            overrides = {}
        self.lambda_fem = overrides.pop('lambda_fem', 0.05)
        super().__init__(cfg, overrides, _callbacks)

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        print("\n[FEM] Replacing C2f modules...")
        replace_c2f_with_fem(model)
        return model

    def criterion(self, preds, batch):
        seg_loss, loss_items = super().criterion(preds, batch)
        fem_loss = collect_fem_loss(self.model)
        total_loss = seg_loss + self.lambda_fem * fem_loss
        return total_loss, loss_items


if __name__ == '__main__':
    yolo = YOLO('yolo11n-seg.pt')
    yolo.train(
        data=r'E:\\mastercode\\6.yolo\\ultralytics-main\\voc20007_seg_yolov8.yaml',
        project=r'E:/mastercode/6.yolo/runs/segment',
        epochs=300,
        imgsz=640,
        batch=8,
        # lr0 = 0.00001,
        # lrf = 0.01,
        # momentum = 0.73,
        # weight_decay = 0.0005,
        # warmup_epochs = 3,
        # optimizer='AdamW',
        device=0  # CPU 则改为 'cpu'
    )



# if __name__ == '__main__':
#     trainer = FEMSegmentationTrainer(
#         overrides={
#             'model':      'yolo11n-seg.pt',
#             'data':       r'E:/mastercode/6.yolo/ultralytics-main/voc20007_seg_yolov8.yaml',
#             'project':    r'E:/mastercode/6.yolo/runs/segment',
#             'epochs':     300,
#             'imgsz':      640,
#             'batch':      8,
#             'device':     0,
#             'lambda_fem': 0.05,
#         }
#     )
#     trainer.train()

New https://pypi.org/project/ultralytics/8.4.22 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.19  Python-3.11.14 torch-2.5.0+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=E:\\mastercode\\6.yolo\\ultralytics-main\\voc20007_seg_yolov8.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-seg.pt, mom

In [1]:
from ultralytics import YOLO
from ultralytics.models.yolo.segment import SegmentationTrainer
from ultralytics.cfg import get_cfg, DEFAULT_CFG
from FEM_DCN import replace_c2f_with_fem, collect_fem_loss

class FEMSegmentationTrainer(SegmentationTrainer):

    def __init__(self, cfg=DEFAULT_CFG, overrides=None, _callbacks=None):
        if overrides is None:
            overrides = {}
        self.lambda_fem = overrides.pop('lambda_fem', 0.05)
        super().__init__(cfg, overrides, _callbacks)

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        print("\n[FEM] Replacing C2f modules...")
        replace_c2f_with_fem(model)
        return model

    def criterion(self, preds, batch):
        seg_loss, loss_items = super().criterion(preds, batch)
        fem_loss = collect_fem_loss(self.model)
        total_loss = seg_loss + self.lambda_fem * fem_loss
        return total_loss, loss_items

if __name__ == '__main__':
    trainer = FEMSegmentationTrainer(
        overrides={
            'model':      'yolo11n-seg.pt',
            'data':       r'E:/mastercode/6.yolo/ultralytics-main/voc20007_seg_yolov8.yaml',
            'project':    r'E:/mastercode/6.yolo/runs/segment',
            'epochs':     300,
            'imgsz':      640,
            'batch':      8,
            'device':     0,
            'lambda_fem': 0.05,
        }
    )
    trainer.train()

Ultralytics 8.4.19  Python-3.11.14 torch-2.5.0+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=E:/mastercode/6.yolo/ultralytics-main/voc20007_seg_yolov8.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train3, nbs=64, nms=False, opset=None, optimize=False, op